In [1]:
# %%
import os
from pathlib import Path

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    GlobalAveragePooling2D
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
# %%
# =========================
# PROJECT CONFIGURATION
# =========================

DATA_DIR = Path(r"D:\Exams\ICT\dataset")

IMG_HEIGHT = 224
IMG_WIDTH = 224
IMAGE_SIZE = (IMG_HEIGHT, IMG_WIDTH)

BATCH_SIZE = 32
EPOCH_COUNT = 15

SAVE_DIR = Path(r"D:\Exams\ICT\models")
SAVE_DIR.mkdir(exist_ok=True)

MODEL_FILE = SAVE_DIR / "transfer_mobilenet_sign_model.keras"

In [3]:
# %%
# =========================
# CLASS FILTERING
# =========================

ignored_classes = {"J", "Z"}

class_names = [
    item.name
    for item in sorted(DATA_DIR.iterdir())
    if item.is_dir() and item.name.upper() not in ignored_classes
]

NUM_CLASSES = len(class_names)

print("Classes used:", class_names)
print("Total number of classes:", NUM_CLASSES)

Classes used: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']
Total number of classes: 24


In [4]:
# %%
# =========================
# DATA AUGMENTATION SETUP
# =========================

data_generator = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=0.20,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    shear_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True
)

In [5]:
# %%
# =========================
# DATA LOADING
# =========================

train_data = data_generator.flow_from_directory(
    directory=str(DATA_DIR),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    classes=class_names,
    shuffle=True,
    seed=42
)

validation_data = data_generator.flow_from_directory(
    directory=str(DATA_DIR),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    classes=class_names,
    shuffle=False,
    seed=42
)

print("Class index mapping:")
print(train_data.class_indices)

Found 3458 images belonging to 24 classes.
Found 863 images belonging to 24 classes.
Class index mapping:
{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'K': 9, 'L': 10, 'M': 11, 'N': 12, 'O': 13, 'P': 14, 'Q': 15, 'R': 16, 'S': 17, 'T': 18, 'U': 19, 'V': 20, 'W': 21, 'X': 22, 'Y': 23}


In [6]:
# %%
# =========================
# TRANSFER LEARNING MODEL
# =========================

input_layer = Input(
    shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    name="image_input"
)

feature_extractor = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_tensor=input_layer
)

feature_extractor.trainable = False

features = feature_extractor.output
features = GlobalAveragePooling2D(name="global_average_pooling")(features)

classifier = Dense(
    units=256,
    activation="relu",
    name="dense_classifier"
)(features)

classifier = Dropout(
    rate=0.5,
    name="dropout_layer"
)(classifier)

output_layer = Dense(
    units=NUM_CLASSES,
    activation="softmax",
    name="prediction_layer"
)(classifier)

transfer_model = Model(
    inputs=input_layer,
    outputs=output_layer,
    name="mobilenetv2_transfer_classifier"
)

C:\Users\HP\AppData\Local\Temp\ipykernel_19632\1482579745.py:11: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  feature_extractor = MobileNetV2(


In [7]:
# %%
# =========================
# MODEL COMPILATION
# =========================

transfer_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

transfer_model.summary()

Model: "mobilenetv2_transfer_classifier"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ image_input[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,592,088 (9.89 MB)

 Trainable params: 334,104 (1.27 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [8]:
# %%
# =========================
# TRAINING
# =========================

training_history = transfer_model.fit(
    train_data,
    validation_data=validation_data,
    epochs=EPOCH_COUNT
)

Epoch 1/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 126s 1s/step - accuracy: 0.5388 - loss: 1.5546 - val_accuracy: 0.8888 - val_loss: 0.5459
Epoch 2/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 118s 1s/step - accuracy: 0.8146 - loss: 0.6035 - val_accuracy: 0.9247 - val_loss: 0.3176
Epoch 3/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - accuracy: 0.8809 - loss: 0.4182 - val_accuracy: 0.8899 - val_loss: 0.3610
Epoch 4/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - accuracy: 0.8959 - loss: 0.3480 - val_accuracy: 0.9548 - val_loss: 0.1721
Epoch 5/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 116s 1s/step - accuracy: 0.9147 - loss: 0.2686 - val_accuracy: 0.9780 - val_loss: 0.1191
Epoch 6/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 129s 1s/step - accuracy: 0.9329 - loss: 0.2350 - val_accuracy: 0.9722 - val_loss: 0.1255
Epoch 7/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 126s 1s/step - accuracy: 0.9320 - loss: 0.2140 - val_accuracy: 0.9849 - val_loss: 0.0784
Epoch 8/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - accuracy: 0.9422 - loss: 0.1909 - val_accu

In [9]:
# %%
# =========================
# SAVE MODEL
# =========================

transfer_model.save(MODEL_FILE)

print(f"✅ Transfer learning model saved at: {MODEL_FILE}")

✅ Transfer learning model saved at: D:\Exams\ICT\models\transfer_mobilenet_sign_model.keras
